# FACE → 완성 정면 캐릭터 → **공식 See-through V3** → PSD

**2026-10-22 공식 저장소 기준 정합 버전 · 실제 GPU 추론 미실행**

- 공식 소스: https://github.com/shitagaki-lab/see-through
- 공식 LayerDiff + Marigold (V3 최대 23개 의미 레이어)만 추론합니다. 별도의 이미지 생성 AI로 파츠를 만들지 않습니다.
- **얼굴만 입력하는 것을 금지**: `full_character_input.png` 완성 캐릭터 한 장 또는 `face__ARTWORK.zip` + 정확히 같은 좌표에 정렬된 `outfit.png` + `hair.png`(+`accessory.png`).
- 노트북이 생성하는 PSD는 **자동 의미 분리 결과**이며, Cubism용 리깅·메시·움직임 합격을 의미하지 않습니다.
- 실행 전 `런타임 → 런타임 유형 변경 → GPU`. 첫 실행은 모델 가중치 다운로드가 필요할 수 있습니다.

**운영 방식:** 환경 설치 → 입력 업로드·합성 → 프리뷰 확인 및 `VISUAL_APPROVED=True` → 공식 스크립트 실행 → PSD와 진단 로그 수집 → ZIP 다운로드.

## 1. 공식 런타임 환경 검사 + 설치

공식 문서의 Python **3.12**, PyTorch **2.8.0+cu128**, torchvision **0.23.0+cu128**, torchaudio **2.8.0+cu128**, `requirements.txt`를 기준으로 합니다. 다른 버전이 반드시 실패한다는 뜻은 아니지만, **공식 동일 설정**으로 판정하지 않습니다.

Python 버전이 다르면 런타임 자체를 Python 3.12로 다시 구성해야 합니다. pip만으로 커널 Python 버전을 바꿀 수 없습니다. PyTorch가 이미 커널에 import되어 설치 버전을 교체했다면 런타임을 재시작하고 처음부터 실행하십시오.

In [ ]:
import sys, subprocess, os, json, shutil, platform, importlib.metadata as md
from pathlib import Path
ROOT=Path('/content')
REPO=ROOT/'see-through'
OUTPUT=ROOT/'face_see_through_v3_output'
OUTPUT.mkdir(parents=True,exist_ok=True)
if not (sys.version_info.major==3 and sys.version_info.minor==12):
    raise RuntimeError(f'OFFICIAL PYTHON MISMATCH: {sys.version.split()[0]} — 공식은 Python 3.12. 3.12 런타임으로 변경하세요.')
if not shutil.which('nvidia-smi'):
    raise RuntimeError('NVIDIA CUDA GPU runtime required (nvidia-smi missing). Colab GPU를 선택하세요.')
subprocess.run(['nvidia-smi'],check=True)
def installed(pkg):
    try:return md.version(pkg)
    except md.PackageNotFoundError:return None
wanted={'torch':'2.8.0+cu128','torchvision':'0.23.0+cu128','torchaudio':'2.8.0+cu128'}
actual={k:installed(k) for k in wanted}
print('Current PyTorch packages:',actual)
if any(actual[k]!=v for k,v in wanted.items()):
    if any(x in sys.modules for x in ('torch','torchvision','torchaudio')):
        raise RuntimeError('PyTorch imported already. Restart runtime before installing pinned official versions.')
    cmd=[sys.executable,'-m','pip','install','--upgrade',*(f'{k}=={v}' for k,v in wanted.items()),'--index-url','https://download.pytorch.org/whl/cu128']
    subprocess.run(cmd,check=True)
    print('Torch packages installed. If Colab already loaded torch internally, restart runtime before inference.')
print('Official Python/PyTorch wheel preflight complete; CUDA runtime health checked after package install.')

## 2. 공식 저장소 설치 + 정확한 버전 기록

`main` 최신 커밋은 바뀔 수 있으므로 SHA를 기록합니다. 설치 전에 사용자가 확인해야 하는 저장소이며, `assets → common/assets` 링크를 공식 지침대로 구성합니다.

In [ ]:
from pathlib import Path
import subprocess,sys,os,json
if not (REPO/'.git').exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/shitagaki-lab/see-through.git',str(REPO)],check=True)
sha=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
print('OFFICIAL_REPO_SHA =',sha)
subprocess.run([sys.executable,'-m','pip','install','-r','requirements.txt'],cwd=REPO,check=True)
assetlink=REPO/'assets'
if not assetlink.exists() and not assetlink.is_symlink():
    assetlink.symlink_to(Path('common/assets'))
if not assetlink.exists():
    raise RuntimeError('Official assets symlink missing or target common/assets unavailable')
import torch
if not torch.cuda.is_available():
    raise RuntimeError('CUDA not available after official package installation. Check driver/runtime and restart.')
VRAM=torch.cuda.get_device_properties(0).total_memory/2**30
print('PyTorch',torch.__version__,'CUDA',torch.version.cuda,'GPU',torch.cuda.get_device_name(0),'VRAM_GiB',round(VRAM,2))
if not (torch.__version__.startswith('2.8.0+cu128')):
    raise RuntimeError(f'PyTorch changed during install: {torch.__version__}; expected official 2.8.0+cu128. Restart/repair before continuing')
if VRAM < 8:
    raise RuntimeError('8 GiB 미만 GPU는 이 노트북의 공식 문서 기반 권장 프로파일 밖입니다.')

## 3. 정면 완성 캐릭터 준비

**정면 마스터만 있는 FACE ZIP은 추론하지 않습니다.** `FACE + OUTFIT + HAIR`가 실제 같은 캐릭터·같은 캔버스·정렬임을 보장해야 합니다. 제공된 FACE ZIP 캔버스는 1536×1920 입니다. 원화 위치 일치는 코드만으로 증명할 수 없으므로 다음 셀 이미지를 시각 확인합니다.

`full_character_input.png` 직접 업로드 시 크기 제한은 공식 구현에 맞춰 유연하게 두지만, 실제로 완성된 캐릭터인지 **사용자가 확인해야** 합니다. RGB 불투명 그림은 공식 스크립트가 사용할 수 있으나 이 제작 파이프라인에서는 투명한 완성 캐릭터 RGBA를 요구합니다.

In [ ]:
import io, zipfile, numpy as np
from PIL import Image
from google.colab import files
from IPython.display import display
uploads=files.upload()
if not uploads:raise ValueError('입력이 없습니다.')
SIZE=(1536,1920)
def strict_rgba(raw,label,required_size=None):
    im=Image.open(io.BytesIO(raw))
    if im.mode!='RGBA':raise ValueError(f'{label}: required RGBA transparent; got {im.mode}')
    if required_size and im.size!=required_size:raise ValueError(f'{label}: {im.size} != {required_size}; auto-resize forbidden')
    a=np.asarray(im.getchannel('A'))
    if a.min()==255 or a.max()==0:raise ValueError(f'{label}: no actual subject/background alpha')
    if im.width<512 or im.height<512:raise ValueError(f'{label}: suspiciously small input')
    return im.copy()
if 'full_character_input.png' in uploads:
    full=strict_rgba(uploads['full_character_input.png'],'full_character_input.png')
    INPUT_MODE='COMPLETE_CHARACTER_UPLOAD'
else:
    req={'face__ARTWORK.zip','outfit.png','hair.png'}
    miss=req-set(uploads)
    if miss:raise ValueError('FACE만으로 See-through 실행 금지. 누락: '+', '.join(sorted(miss)))
    with zipfile.ZipFile(io.BytesIO(uploads['face__ARTWORK.zip'])) as z:
        face=strict_rgba(z.read('outputs/master.png'),'FACE master',SIZE)
    outfit=strict_rgba(uploads['outfit.png'],'OUTFIT',SIZE)
    hair=strict_rgba(uploads['hair.png'],'HAIR',SIZE)
    full=Image.new('RGBA',SIZE,(0,0,0,0))
    for im in (outfit,face,hair):full.alpha_composite(im)
    if 'accessory.png' in uploads:
        full.alpha_composite(strict_rgba(uploads['accessory.png'],'ACCESSORY',SIZE))
    INPUT_MODE='COMPOSITE_OUTFIT_FACE_HAIR'
input_path=ROOT/'full_character_input.png'
full.save(input_path)
print('INPUT_MODE:',INPUT_MODE,'Size:',full.size,'Alpha extrema:',full.getchannel('A').getextrema())
display(full.copy().resize((400,int(400*full.height/full.width))))
print('검수: 눈·머리카락·목·옷깃, 헤어라인, 몸통과 목이 올바로 겹치며 완전한 캐릭터인가?')

## 4. 필수 육안 승인

**아래 불리언을 `True`로 변경하는 것은 실제 프리뷰를 눈으로 확인한 경우에 한합니다.** 1536×1920 동일 크기라는 사실만으로 픽셀 위치·캐릭터 정합이 증명되지는 않습니다.

In [ ]:
VISUAL_APPROVED=False  # 위 프리뷰를 보고 완성된 캐릭터임을 확인한 뒤 True 로 변경
if not VISUAL_APPROVED:
    raise RuntimeError('Full character/alignment preview not approved. Review preview before inference.')

## 5. 공식 CLI 플래그와 VRAM 프로파일 검사

공식 인자 존재 여부를 `ast`로 파싱해 **모델 로딩 전에** 확인합니다. 스크립트가 업데이트되어 인터페이스가 바뀌었으면 진행을 중단합니다. `--tblr_split` 좌우 눈/귀 등의 분리를 시도하나 성공 보장은 아닙니다.

In [ ]:
import ast
from pathlib import Path
assert VISUAL_APPROVED
if VRAM >= 16:
    MODE='bf16'; script='inference/scripts/inference_psd.py'; resolution=1280; offload=False
elif VRAM >= 10:
    MODE='bf16_group_offload'; script='inference/scripts/inference_psd.py'; resolution=1280; offload=True
else:
    MODE='nf4'; script='inference/scripts/inference_psd_quantized.py'; resolution=1024 if VRAM<9 else 1280; offload=False
    subprocess.run([sys.executable,'-m','pip','install','-r','requirements-inference-bnb.txt'],cwd=REPO,check=True)
script_file=REPO/script
if not script_file.exists():raise FileNotFoundError(f'Official script missing: {script_file}')
source=script_file.read_text(encoding='utf-8')
tree=ast.parse(source)
flags=set()
for node in ast.walk(tree):
    if isinstance(node,ast.Call) and isinstance(node.func,ast.Attribute) and node.func.attr=='add_argument':
        for v in node.args:
            if isinstance(v,ast.Constant) and isinstance(v.value,str) and v.value.startswith('--'):flags.add(v.value)
required={'--srcp','--save_to_psd','--save_dir','--resolution','--tblr_split'}
if offload:required.add('--group_offload')
missing=required-flags
if missing:raise RuntimeError(f'OFFICIAL CLI CHANGED; missing {missing} in {script_file}. Halt before GPU inference.')
SAVE_DIR=OUTPUT/'official_output'
SAVE_DIR.mkdir(parents=True,exist_ok=True)
cmd=[sys.executable,script,'--srcp',str(input_path),'--save_to_psd','--save_dir',str(SAVE_DIR),'--tblr_split','--resolution',str(resolution),'--seed','42']
if offload:cmd.append('--group_offload')
run_meta={'official_git_sha':sha,'mode':MODE,'vram_gib':round(VRAM,2),'resolution':resolution,'command':cmd,
          'input_mode':INPUT_MODE,'full_character_preview_human_approved':VISUAL_APPROVED,
          'cubism_verified':False,'see_through_psd_verified':False}
(OUTPUT/'run_meta.json').write_text(json.dumps(run_meta,ensure_ascii=False,indent=2))
print('Official profile',MODE,'script',script,'resolution',resolution,'flags passed')

## 6. GPU 추론: 원본 공식 파이프라인만 실행

기본 스크립트가 **두 단계 LayerDiff + Marigold + PSD**를 실행합니다. 추론 중 오류는 숨기지 않고 로그를 보존합니다. 처음 가중치 다운로드 시 시간이 더 소요될 수 있습니다.

In [ ]:
import time, subprocess
start=time.monotonic()
print('RUNNING:', ' '.join(cmd))
with (OUTPUT/'see_through_stdout.log').open('w',encoding='utf-8') as out, (OUTPUT/'see_through_stderr.log').open('w',encoding='utf-8') as err:
    proc=subprocess.run(cmd,cwd=REPO,stdout=out,stderr=err,text=True)
run_meta['exit_code']=proc.returncode
run_meta['elapsed_seconds']=round(time.monotonic()-start,2)
(OUTPUT/'run_meta.json').write_text(json.dumps(run_meta,ensure_ascii=False,indent=2))
if proc.returncode:
    print((OUTPUT/'see_through_stderr.log').read_text(encoding='utf-8')[-5000:])
    raise RuntimeError(f'See-through failed with exit {proc.returncode}; diagnostics in {OUTPUT}')
psds=list(SAVE_DIR.rglob('*.psd'))
if not psds:raise RuntimeError('Exit code 0 but no official PSD produced; not a successful run')
print('Found official PSDs:',[str(p) for p in psds])

## 7. PSD 구조/이미지 실체 검사 + 로그·ZIP 다운로드

이 검사는 **PSD가 실제 여러 레이어로 열리는지**만 확인합니다. 눈 뒤 흰자와 입안 인페인팅 품질·Live2D 메시·움직임은 별도 육안 및 Cubism 검증이 필수입니다.

In [ ]:
from psd_tools import PSDImage
import hashlib,zipfile,json,shutil
qa={'repo_sha':sha,'mode':MODE,'psds':[],'cubism_test':'NOT_RUN','rigging_usable':'UNVERIFIED'}
def digest(path):
    d=hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda:f.read(2**20),b''):d.update(block)
    return d.hexdigest()
for p in psds:
    psd=PSDImage.open(p)
    nodes=[]
    def inspect(items,depth=0):
        for a in items:
            nodes.append({'name':str(a.name),'depth':depth,'visible':bool(a.is_visible()),'group':bool(a.is_group())})
            if a.is_group():inspect(a,depth+1)
    inspect(psd)
    non_groups=[v for v in nodes if not v['group']]
    if len(non_groups)<2:raise RuntimeError(f'PSD appears flattened: {p}')
    if psd.width<128 or psd.height<128:raise RuntimeError(f'Invalid PSD dimensions: {p}')
    qa['psds'].append({'file':str(p.relative_to(SAVE_DIR)),'bytes':p.stat().st_size,'size':list(psd.size),
       'leaf_layers':len(non_groups),'layers':nodes,'sha256':digest(p)})
qa['structure_pass']=bool(qa['psds'])
qa['remaining_human_checks']=['Eye whites exposed when pupils move','Top/bottom lashes and lid coverage during blinks',
                              'Mouth interior during talking','Deformer and ArtMesh in Cubism','45° turn mesh does not reveal gaps']
(OUTPUT/'PSD_STRUCTURE_QA.json').write_text(json.dumps(qa,ensure_ascii=False,indent=2),encoding='utf-8')
shutil.copy2(input_path,OUTPUT/'full_character_input.png')
run_meta['see_through_psd_verified']=True
(OUTPUT/'run_meta.json').write_text(json.dumps(run_meta,ensure_ascii=False,indent=2))
archive=ROOT/'SeeThrough_OFFICIAL_GPU_RESULT_v3.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=1) as z:
    for p in OUTPUT.rglob('*'):
        if p.is_file():z.write(p,p.relative_to(OUTPUT))
print('OFFICIAL PSD STRUCTURE PASS:',len(qa['psds']),'files')
from google.colab import files
files.download(str(archive))